# Set up

In [2]:
import numpy as np
import torch
import gpytorch
import matplotlib.pyplot as plt
from gpytorch.constraints import Interval

from scipy.integrate import odeint
import pandas as pd
from math import pi

import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import functools
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
from tqdm.notebook import trange

import tqdm as tqdm

import linear_operator

from scipy.integrate import odeint
from scipy.signal import find_peaks

from sklearn.preprocessing import StandardScaler

from typing import Dict, Any, Callable, Tuple

# Data

## LV

In [3]:
train_x = np.load("Data/LV_train_x.npy")    # 形状 (M, 2)
train_y = np.load("Data/LV_train_y.npy")    # 形状 (M, 2*K)

# GP Model

In [3]:
device = 'cpu'

In [4]:
X_train = torch.tensor(train_x, dtype=torch.float32)
Y_train = torch.tensor(train_y, dtype=torch.float32)

X_train = X_train.to(device)
Y_train = Y_train.to(device)

## MGP

In [5]:
class MultitaskGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(MultitaskGPModel, self).__init__(train_x, train_y, likelihood)
        # self.mean_module = gpytorch.means.MultitaskMean(
        #     gpytorch.means.ConstantMean(), num_tasks=train_y.shape[1]
        # )
        self.mean_module = gpytorch.means.MultitaskMean(
            gpytorch.means.ZeroMean(), num_tasks=train_y.shape[1]
        )
        self.covar_module = gpytorch.kernels.MultitaskKernel(
            gpytorch.kernels.RBFKernel(), num_tasks=train_y.shape[1], rank=1
        )

        # base_rbf = gpytorch.kernels.RBFKernel()
        # self.covar_module = gpytorch.kernels.MultitaskKernel(base_rbf, num_tasks=train_y.shape[1], rank=1)

        # ls_init=1
        # self.covar_module.data_covar_module.lengthscale = torch.as_tensor(float(ls_init)).view(1, 1)



    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultitaskMultivariateNormal(mean_x, covar_x)

## BanchIndependentGP

In [6]:
class BatchIndependentGP(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(BatchIndependentGP, self).__init__(train_x, train_y, likelihood)
        # self.mean_module = gpytorch.means.ConstantMean(batch_shape=torch.Size([train_y.shape[1]]))
        self.mean_module = gpytorch.means.ZeroMean(batch_shape=torch.Size([train_y.shape[1]]))
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel(ard_num_dims=train_x.size(-1), batch_shape=torch.Size([train_y.shape[1]])),
            batch_shape=torch.Size([train_y.shape[1]])
        )

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultitaskMultivariateNormal.from_batch_mvn(
            gpytorch.distributions.MultivariateNormal(mean_x, covar_x)
        )

## Training GP

In [7]:
lr=0.05
num_iterations=5000
patience=10
disable_progbar=False

In [8]:
likelihood = gpytorch.likelihoods.MultitaskGaussianLikelihood(num_tasks=Y_train.shape[1])
# model = MultitaskGPModel(X_train, Y_train, likelihood)
model = BatchIndependentGP(X_train, Y_train, likelihood)

model = model.to(device)
likelihood = likelihood.to(device)

model.train()
likelihood.train()

MultitaskGaussianLikelihood(
  (raw_task_noises_constraint): GreaterThan(1.000E-04)
  (raw_noise_constraint): GreaterThan(1.000E-04)
)

In [9]:
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

best_loss = float('inf')
counter = 0
iterator = tqdm.tqdm(range(num_iterations), disable=disable_progbar)

for i in iterator:
    optimizer.zero_grad()
    output = model(X_train)
    loss = -mll(output, Y_train)
    loss.backward()
    if not disable_progbar:
        iterator.set_postfix(loss=loss.item())
    optimizer.step()

    if loss <= best_loss:
        best_loss = loss
        best_state = model.state_dict()  
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            model.load_state_dict(best_state)  
            break

 33%|███▎      | 1664/5000 [00:12<00:24, 133.71it/s, loss=1.14]


# Tool

In [ ]:
def _clear_all_caches(module: torch.nn.Module):
    """清理 gpytorch 的 memoize 缓存，避免 dtype/device 切换后出现旧缓存。"""
    try:
        gpytorch.utils.memoize.clear_cache(module)
    except Exception:
        pass
    for m in module.modules():
        try:
            gpytorch.utils.memoize.clear_cache(m)
        except Exception:
            pass


def _to_numpy(x):
    return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)

def _dev_dt_from(model, fallback_device="cpu"):
    """从模型抓取 device/dtype；若失败则回退。"""
    try:
        p = next(model.parameters())
        return p.device, p.dtype
    except StopIteration:
        return torch.device(fallback_device), torch.float32
    

def to_dense_safe(op: torch.Tensor) -> torch.Tensor:
    """兼容 linear_operator 和老 LazyTensor 的安全稠密化。"""
    if isinstance(op, torch.Tensor):
        return op
    if hasattr(op, "to_dense"):
        return op.to_dense()
    if hasattr(op, "evaluate"):
        return op.evaluate()
    raise TypeError(f"Object of type {type(op)} is not convertible to dense.")



# Dim

X = (x_1, ... x_n)  (N,D)

Y = (y_1, ... y_n)  (N,P)


In [11]:
train_x.shape

(94, 2)

In [12]:
train_y.shape

(94, 4)

In [37]:
dev, dt = train_x.device, train_x.dtype

# Test point

In [13]:
start_points = torch.tensor([[1.3, 1.2]], dtype=torch.float32)

# K_XX = K(X,X)  (N,N)

## Non-expr

In [ ]:
X = model.train_inputs[0]

K_batch_lazy = model.covar_module(X)   # LinearOperator, shape: P × N × N
# K_batch_lazy = model.covar_module.data_covar_module(X)   # LinearOperator, shape: P × N × N

K_batch = to_dense_safe(K_batch_lazy)    # torch.Tensor, shape: P × N × N


K_full_lazy = linear_operator.operators.BlockDiagLinearOperator(K_batch_lazy)  # (PN)×(PN)
K_XX = to_dense_safe(K_full_lazy)


mvn = model(X)                     # MultitaskMVN
K_ff = likelihood(mvn).covariance_matrix  # (PN)×(PN) with noise

In [21]:
K_XX.shape

torch.Size([376, 376])

# K_XJ

dK(x',X)/dx'   (N,D)

## Autograd

In [14]:
x_flat = start_points.squeeze(0).detach().requires_grad_(True)    # [D]

def g(x_flat_):
    x1 = x_flat_.unsqueeze(0)                          # [1, D]
    k_row = to_dense_safe(model.covar_module(x1, X_train)).squeeze(0)  # [N]
    # k_row = to_dense_safe(model.covar_module.data_covar_module(x1, X_train)).squeeze(0)  # [N]
    return k_row

try:
    J = torch.autograd.functional.jacobian(g, x_flat, vectorize=True)  # [N, D] ; shape: (P, n*, N)
except TypeError:
    J = torch.autograd.functional.jacobian(g, x_flat)                   # [N, D] ; shape: (P, n*, N)
dk_dx = J

In [17]:
to_dense_safe(model.covar_module(start_points, X_train)).shape

torch.Size([4, 1, 94])

In [15]:
dk_dx.shape

torch.Size([4, 1, 94, 2])

In [34]:
blocks = [dk_dx[p].reshape(dk_dx.size(1), -1) for p in range(dk_dx.size(0))]
K_blockdiag = torch.block_diag(*blocks)

In [37]:
K_blockdiag.shape

torch.Size([4, 752])

In [ ]:
K_blockdiag

## Expression

In [33]:
model.covar_module.base_kernel.lengthscale

tensor([[[5.2501, 0.0515]],

        [[1.5602, 0.0721]],

        [[0.0307, 0.0488]],

        [[0.6750, 0.6966]]], grad_fn=<SoftplusBackward0>)

In [ ]:
K_ZX = model.covar_module(start_points, X).to_dense()  # (P, M, N)
P, M, N = K_ZX.shape

In [23]:
covar = model.covar_module
base = covar.base_kernel if isinstance(covar, gpytorch.kernels.ScaleKernel) else covar
# base.lengthscale: (P, 1, D) or (1, 1, D)
ell2 = (base.lengthscale.squeeze(-2) ** 2)     # -> (P, D) or (1, D)
ell2 = ell2.unsqueeze(1).unsqueeze(2)          # -> (P, 1, 1, D)

# ∂/∂z k(z, x) = k(z,x) * (x - z) / ℓ^2

Xb = X.unsqueeze(0)                            # 1 × N × D
Zb = start_points.unsqueeze(0)                            # 1 × M × D
diff_XminusZ = Xb.unsqueeze(1) - Zb.unsqueeze(2)   # 1 × M × N × D -> (X - Z)
diff_XminusZ = diff_XminusZ.expand(P, -1, -1, -1)  # P × M × N × D

# K_JX: (P, M, N, D)
K_JX_4d = K_ZX.unsqueeze(-1) * (diff_XminusZ / ell2)

In [24]:
K_JX_4d.shape

torch.Size([4, 1, 94, 2])

In [25]:
dk_dx.shape

torch.Size([4, 1, 94, 2])

# (K_{XX}+Σ)^{-1} y

In [18]:
model.eval(); likelihood.eval()

X = model.train_inputs[0]                 # (N, D)
Z = start_points.to(X.device, X.dtype)               # (M, D)


with gpytorch.settings.fast_pred_var():
    _ = likelihood(model(X))

ps = model.prediction_strategy
alpha_flat = ps.mean_cache

d:\anaconda3\envs\GPTG\Lib\site-packages\gpytorch\models\exact_gp.py:296: GPInputWarning: The input matches the stored training data. Did you forget to call model.train()?
  warnings.warn(


In [20]:
alpha_flat.shape

torch.Size([376])

In [21]:
alpha = alpha_flat.view(P, N)

In [22]:
alpha.shape

torch.Size([4, 94])

In [35]:
alpha

tensor([[ 5.6191e-01, -1.5402e+00, -2.3768e-02, -3.8779e-01,  1.6195e+00,
         -1.1944e+00,  2.3598e+00,  9.7163e-01,  1.7061e-01, -1.5015e+00,
         -1.4850e+01, -6.6209e-01,  1.2000e+00, -1.2221e+00,  3.3773e+01,
          6.7891e-01,  1.8946e+00, -5.9796e-01, -2.4921e+01,  1.4858e+00,
          1.6697e+00,  7.5722e-01,  5.4239e+00,  1.1906e+00,  3.9879e-01,
          1.7769e+00, -1.1200e+00,  2.8467e-01, -2.1470e-01, -1.4353e+00,
         -1.4837e+01, -9.5403e-01,  6.6269e-01, -1.3052e+00,  3.1488e+01,
          1.0403e-01,  1.5711e+00, -7.0963e-01, -1.2238e+01,  1.3321e+00,
          1.6277e+00,  4.0425e-01, -1.7970e+01,  1.3567e+00,  4.8593e-01,
          1.6313e+00,  1.2329e+01,  5.0262e-01, -5.3444e-01,  1.6681e+00,
          2.9436e+00, -2.8628e-01, -1.0860e+00,  9.3547e-01, -2.7466e+00,
         -9.1053e-01, -1.2610e+00,  1.5700e-02,  8.9528e-02, -1.3438e+00,
          1.1483e+00, -9.4057e-01,  1.7331e+01,  9.1977e-01,  1.5815e+00,
         -1.3569e-01, -2.9432e+01,  1.

# E(J|Y) = KJX​(KXX​+Σ)−1y

In [ ]:
KJX_mat = dk_dx.reshape(P, M * X.size(-1), N)      # (P, D, N)

In [27]:
KJX_mat.shape

torch.Size([4, 2, 94])

In [ ]:
jac_flat = torch.bmm(KJX_mat, alpha.unsqueeze(-1)).squeeze(-1)  # (P, D)

In [29]:
jac_flat.shape

torch.Size([4, 2])

In [33]:
jac_flat

tensor([[ 3.4344e-01,  1.3109e-01],
        [-1.1366e-03,  1.2359e-03],
        [ 7.8567e+02, -5.2198e+02],
        [-2.9236e-05, -2.0679e-06]])

In [30]:
jac_mean = jac_flat.view(P, M, X.size(-1))

In [34]:
jac_mean

tensor([[[ 3.4344e-01,  1.3109e-01]],

        [[-1.1366e-03,  1.2359e-03]],

        [[ 7.8567e+02, -5.2198e+02]],

        [[-2.9236e-05, -2.0679e-06]]])